In [1]:
!pip install albumentations -q

In [5]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from tqdm.notebook import tqdm
import albumentations as A
import warnings
import yaml
warnings.filterwarnings('ignore')

In [9]:
DATASET_ROOT = r"./VisDrone_Dataset"

with open(f"{DATASET_ROOT}/visdrone.yaml", "r") as f:
    data = yaml.safe_load(f)

# ── Change this to wherever you unzipped the dataset ──
DATASET_ROOT = data["path"]
# r"..." = raw string, prevents Windows backslashes from causing issues
# Example: r"C:\Users\John\Desktop\VisDrone_Dataset"

TRAIN_IMG_PATH = os.path.join(DATASET_ROOT, data["train"])
TRAIN_LBL_PATH = os.path.join(os.path.dirname(TRAIN_IMG_PATH), "labels")
VAL_IMG_PATH   = os.path.join(DATASET_ROOT, data["val"])
VAL_LBL_PATH   = os.path.join(os.path.dirname(VAL_IMG_PATH), "labels")
TEST_IMG_PATH  = os.path.join(DATASET_ROOT, data["test"])

# Folder where all output charts/images will be saved
OUTPUT_DIR = os.path.join(DATASET_ROOT, "task01_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
# exist_ok=True → no error if folder already exists

# This Kaggle version has no "ignored" class
# All IDs are shifted down by 1 compared to original VisDrone

CLASS_NAMES = {int(k): v for k, v in data["names"].items()}

print(CLASS_NAMES)

# We only care about humans and cars for this project
# humans = pedestrian (0) + people (1)
# cars   = car (3)
TARGET_CLASSES = {0: "pedestrian", 1: "people", 3: "car"}

# Verify all paths exist before doing anything
print("Checking paths...")
for name, path in [
    ("Train Images",  TRAIN_IMG_PATH),
    ("Train Labels",  TRAIN_LBL_PATH),
    ("Val Images",    VAL_IMG_PATH),
    ("Val Labels",    VAL_LBL_PATH),
]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND — fix DATASET_ROOT"
    print(f"  {status}  {name}: {path}")

{0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car', 4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle', 8: 'bus', 9: 'motor'}
Checking paths...
  ✅  Train Images: ./VisDrone_Dataset\VisDrone2019-DET-train/images
  ✅  Train Labels: ./VisDrone_Dataset\VisDrone2019-DET-train\labels
  ✅  Val Images: ./VisDrone_Dataset\VisDrone2019-DET-val/images
  ✅  Val Labels: ./VisDrone_Dataset\VisDrone2019-DET-val\labels


In [10]:
def parse_annotation_file(lbl_path, img_w, img_h):
    """
    Reads one YOLO format .txt label file.
    
    YOLO format — each line has 5 values separated by spaces:
        class_id  cx  cy  w  h
    
    class_id → integer class index (0–9)
    cx, cy   → center of bounding box, normalized between 0.0 and 1.0
               (fraction of image width and height)
    w, h     → width and height of bounding box, also normalized 0.0–1.0
    
    Example line: "0 0.482 0.271 0.109 0.215"
    Means: class 0 (pedestrian), center at 48.2% from left, 27.1% from top,
           box is 10.9% of image wide and 21.5% of image tall
    
    We convert to pixel values by multiplying by image dimensions.
    
    Unlike original VisDrone, there is NO score, occlusion, or truncation field.
    """
    
    objects = []
    
    with open(lbl_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        # Remove newline characters and leading/trailing spaces
        
        if not line:
            continue
        # Skip empty lines
        
        parts = line.split()
        # Split by whitespace (spaces)
        # "0 0.482 0.271 0.109 0.215" → ['0', '0.482', '0.271', '0.109', '0.215']
        
        if len(parts) < 5:
            continue
        # Skip malformed lines
        
        cat_id = int(parts[0])
        cx_norm = float(parts[1])   # center x as fraction of image width
        cy_norm = float(parts[2])   # center y as fraction of image height
        cw_norm = float(parts[3])   # box width as fraction of image width
        ch_norm = float(parts[4])   # box height as fraction of image height
        
        # Convert normalized values to actual pixel values
        cx_px = cx_norm * img_w
        cy_px = cy_norm * img_h
        w_px  = cw_norm * img_w
        h_px  = ch_norm * img_h
        # Multiply fraction by image dimension to get pixels
        # e.g., cw=0.109, img_w=1920 → 0.109 × 1920 = 209.3 pixels wide
        
        # Convert from center format to top-left corner format
        x_px = cx_px - w_px / 2
        y_px = cy_px - h_px / 2
        # YOLO stores center point, but for drawing boxes we need top-left corner
        
        objects.append({
            'category': cat_id,
            'x': x_px,      # top-left x in pixels
            'y': y_px,      # top-left y in pixels
            'w': w_px,      # width in pixels
            'h': h_px,      # height in pixels
            'cx_norm': cx_norm,   # keep normalized values too for analysis
            'cy_norm': cy_norm,
            'cw_norm': cw_norm,
            'ch_norm': ch_norm,
        })
    
    return objects